# OralVerse — MeshSegNet Training

**Run this notebook on Google Colab or Kaggle (free GPU).**

| Platform | GPU | VRAM | Time estimate | Cost |
|---|---|---|---|---|
| Google Colab free | T4 | 15 GB | ~5–7 hrs | Free |
| Kaggle | P100 | 16 GB | ~4–5 hrs | Free (30 hrs/wk) |
| Colab Pro | A100 | 40 GB | ~2 hrs | $10/mo |

**At the end:** download `meshsegnet_upper_best.pt` and `meshsegnet_lower_best.pt` from the Files panel.

---

### Colab setup
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells (`Runtime → Run all`)
3. Come back in ~6 hours
4. Download the `.pt` files from the left Files panel

### Kaggle setup  
1. New Notebook → Settings → Accelerator: GPU P100
2. Paste each cell, run all
3. Output files appear in `/kaggle/working/checkpoints/`

## 1. Check GPU

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

## 2. Clone repo + install deps

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/OralVerse-1.git"  # ← change this
BRANCH   = "claude/features"
WORK_DIR = "/content/OralVerse-1"  # /content for Colab, /kaggle/working for Kaggle

# Detect Kaggle
if os.path.exists("/kaggle"):
    WORK_DIR = "/kaggle/working/OralVerse-1"

if not os.path.exists(WORK_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q trimesh==4.4.3 tqdm==4.66.5 scikit-learn==1.5.2
print("Dependencies installed.")

## 3. Download dataset

Downloads only **01.zip** (~2 GB, ~450 scans) by default — enough for a good model.
Set `DOWNLOAD_ALL = True` to get all 1800 scans (8 GB, better accuracy).

In [ ]:
DOWNLOAD_ALL = False  # set True for full dataset (8GB, ~4x longer training)

RAW_DIR  = f"{WORK_DIR}/ai/orthodontics/segmentation/meshsegnet/raw_data"
DATA_DIR = f"{WORK_DIR}/ai/orthodontics/segmentation/meshsegnet/data"
CKPT_DIR = f"{WORK_DIR}/ai/orthodontics/segmentation/meshsegnet/checkpoints"

!mkdir -p {RAW_DIR} {DATA_DIR} {CKPT_DIR}

BASE = "https://zenodo.org/record/7151927/files"
parts = ["01", "02", "03", "04"] if DOWNLOAD_ALL else ["01"]

for part in parts:
    dest = f"{RAW_DIR}/{part}.zip"
    if not os.path.exists(dest):
        print(f"Downloading {part}.zip ...")
        !curl -L --progress-bar -o {dest} "{BASE}/{part}.zip?download=1"
    else:
        print(f"[skip] {part}.zip already downloaded")
    print(f"Extracting {part}.zip ...")
    !unzip -q -o {dest} -d {RAW_DIR}

print("\nDataset ready.")
!find {RAW_DIR} -name "*.obj" | wc -l

## 4. Preprocess scans → .npz

In [ ]:
!python3 -m ai.orthodontics.segmentation.meshsegnet.prepare_data \
    --data_dir {RAW_DIR} \
    --out_dir  {DATA_DIR}

print("\nPreprocessing complete.")
!find {DATA_DIR} -name "*.npz" | wc -l

## 5. Train — Upper arch

In [ ]:
EPOCHS    = 50    # 50 is enough for a solid model; use 100 for best quality
MAX_FACES = 12000 # reduce if you get OOM errors on T4

!python3 -m ai.orthodontics.segmentation.meshsegnet.train \
    --data_dir  {DATA_DIR} \
    --arch      upper \
    --out_dir   {CKPT_DIR} \
    --epochs    {EPOCHS} \
    --lr        1e-3 \
    --max_faces {MAX_FACES}

## 6. Train — Lower arch

In [ ]:
!python3 -m ai.orthodontics.segmentation.meshsegnet.train \
    --data_dir  {DATA_DIR} \
    --arch      lower \
    --out_dir   {CKPT_DIR} \
    --epochs    {EPOCHS} \
    --lr        1e-3 \
    --max_faces {MAX_FACES}

## 7. Evaluate

In [ ]:
for arch in ["upper", "lower"]:
    ckpt = f"{CKPT_DIR}/meshsegnet_{arch}_best.pt"
    if os.path.exists(ckpt):
        !python3 -m ai.orthodontics.segmentation.meshsegnet.evaluate \
            --checkpoint {ckpt} \
            --data_dir   {DATA_DIR} \
            --split      test
    else:
        print(f"No checkpoint found for {arch}")

## 8. Download checkpoints

**Colab**: Run the cell below to get download links.  
**Kaggle**: Files appear in `/kaggle/working/checkpoints/` — click the output tab.

In [ ]:
import os
from pathlib import Path

checkpoints = list(Path(CKPT_DIR).glob("*.pt"))

if not checkpoints:
    print("No checkpoints found — check training logs above for errors.")
else:
    print(f"Found {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / 1e6
        print(f"  {ckpt.name}  ({size_mb:.1f} MB)")

    # Colab download
    try:
        from google.colab import files
        print("\nDownloading via Colab...")
        for ckpt in checkpoints:
            files.download(str(ckpt))
    except ImportError:
        # Kaggle — files already accessible in output
        print("\nKaggle: access files from the Output tab on the right.")

print("\nAfter downloading, place the .pt files in:")
print("  ai/orthodontics/segmentation/meshsegnet/checkpoints/")
print("\nThen verify with:")
print("  python3 -m ai.orthodontics.segmentation.meshsegnet.export_model \\")
print("    --checkpoint ai/orthodontics/segmentation/meshsegnet/checkpoints/meshsegnet_upper_best.pt")
print("\nTo activate in production:")
print("  ORALVERSE_SEGMENTER=meshsegnet")
print("  MESHSEGNET_WEIGHTS=/path/to/meshsegnet_upper_best.pt")

---

## If you have Apple Silicon (M1/M2/M3) — train locally instead

PyTorch supports MPS (Metal GPU) on Apple Silicon. Slower than a cloud GPU but free and no session limits.

```bash
# Check MPS is available
python3 -c "import torch; print(torch.backends.mps.is_available())"

# Train (MPS is auto-detected in train.py via torch.device('mps'))
python3 -m ai.orthodontics.segmentation.meshsegnet.train \
    --data_dir ai/orthodontics/segmentation/meshsegnet/data \
    --arch upper --epochs 50 --max_faces 8000
```

Expected time on M1 Pro: ~12–16 hrs for 50 epochs on 450 scans.